# 02 — Grafo de conocimiento en Relate

Genera el Cypher de carga desde las 31 tripletas semilla, instala el servidor **Relate** (Neo4j, en memoria) dentro de la instancia de Colab, carga el grafo y corre las 12 preguntas en Cypher. Todo dentro de Colab: nada de pasos de máquina local.

**Prerrequisito:** notebook 01 (chunks + tripletas semilla).

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

## Dependencia: driver neo4j

`pipeline/load_relate.py` importa `neo4j` (lazy import): sin el paquete, la carga por Bolt falla. Se instala aquí, antes de cargar el grafo.

In [ ]:
%pip install -q neo4j>=6.4.1

## Datos procesados: primero en Drive, si no, súbelos

Este notebook necesita `pipeline/data/chunks.jsonl` (las 151 páginas del informe extraídas). El flujo es:

1. **Buscarlo en Drive** (ruta estándar, bajo el repo clonado).
2. **Si no está, subirlo** con el widget de Colab de la celda siguiente: el archivo es `chunks.jsonl`. Lo consigues de una corrida anterior en Drive o generándolo del repo con `make extract` (`python pipeline/extract.py`).
3. **Solo si no lo subes**, generarlo aquí corriendo `extract.py` (descarga el PDF ~19 MB y extrae las 151 páginas, ~1–2 min). Es el último recurso, no el camino principal.

`triples_seed.json` y `eval_questions.json` vienen con el clon del repo: no hace falta subirlos.

In [ ]:
import json, subprocess, sys
from pathlib import Path

CHUNKS = REPO + "/pipeline/data/chunks.jsonl"
DATA_DIR = REPO + "/pipeline/data"


def _chunks_ok(path):
    try:
        with open(path) as f:
            first = f.readline()
            n = 1 + sum(1 for _ in f)
        row = json.loads(first)
        return n >= 100 and isinstance(row.get("text"), str) and len(row["text"]) > 0
    except Exception:
        return False


if _chunks_ok(CHUNKS):
    print(f"datos procesados OK en Drive: {CHUNKS}")
else:
    print("chunks.jsonl NO está en Drive.")
    print("Súbelo con el widget de abajo (botón 'Elegir archivos').")
    print("Qué subir: chunks.jsonl — las 151 páginas del informe FEMSA 2024.")
    print("Dónde conseguirlo: de una corrida anterior en Drive, o del repo con `make extract`.")
    print("Si no lo subes, el notebook lo genera con extract.py (último recurso).")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = Path(DATA_DIR) / Path(name).name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"subido: {dest} ({len(data)} bytes)")
    if not _chunks_ok(CHUNKS) and len(uploaded) == 1:
        # El archivo subido puede venir con otro nombre: probar como chunks.jsonl.
        only = next(iter(uploaded.values()))
        Path(CHUNKS).write_bytes(only)
        print("probando el archivo subido como chunks.jsonl...")
    if _chunks_ok(CHUNKS):
        print("datos listos por subida.")
    else:
        print("Sin subida válida: último recurso, generar con extract.py...")
        print("(descarga el PDF ~19 MB y extrae las 151 páginas, ~1-2 min)")
        try:
            import pypdf
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "pypdf>=7.1.1"], check=True)
        r = subprocess.run([sys.executable, "extract.py"], cwd=REPO + "/pipeline")
        print("extract.py exit:", r.returncode)
        assert _chunks_ok(CHUNKS), "No se pudo obtener chunks.jsonl."
    print(f"chunks.jsonl listo: {CHUNKS}")

## Generar el Cypher de carga

Corre `pipeline/build_cypher.py` tal cual: genera `load_graph.cypher` (MERGE idempotentes) desde `triples_seed.json`. No se cargan tripletas extraídas sin revisar — esa regla sigue vigente aquí.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "build_cypher.py"], cwd=REPO + "/pipeline")
print("exit:", r.returncode)

## Instalar Relate en la instancia

Relate publica binarios Linux x86_64 (`relate-linux-amd64`, versión fijada v1.2.0, la misma que se verificó en local). El instalador oficial (`curl -sSfL https://relatedb.ai/install.sh | bash`) resuelve el último release estable y verifica la firma; si el binario ya existe, no se reinstala. El binario queda en `~/.local/bin/relate` **dentro de la instancia** (se pierde al reiniciar la sesión: hay que re-correr esta celda).

In [ ]:
%%bash
if [ -x "$HOME/.local/bin/relate" ]; then
  echo "relate ya instalado: $($HOME/.local/bin/relate --version | head -1)"
else
  echo "instalando Relate..."
  curl -sSfL https://relatedb.ai/install.sh | bash
  "$HOME/.local/bin/relate" --version | head -1
fi
export PATH="$HOME/.local/bin:$PATH"
relate --version | head -1

## Arrancar el servidor Relate en segundo plano

El store `data/femsa.rl` vive en el repo (en Drive, así que persiste). El servidor escucha solo en localhost, sin autenticación (`--auth none`), igual que en local. Si la sesión de Colab se reinicia, hay que re-correr esta celda: el store no se pierde, solo el proceso.

In [ ]:
import os, subprocess, time

RELATE_BIN = os.path.expanduser("~/.local/bin/relate")
DB = REPO + "/pipeline/data/femsa.rl"
LOG = open("relate.log", "w")

def relate_up():
    try:
        import urllib.request
        with urllib.request.urlopen("http://localhost:7687", timeout=2) as r:
            return True
    except Exception:
        return False

# Bolt (7687) no responde a HTTP, asi que la prueba es: ¿el proceso vive y el puerto escucha?
import socket
def port_open(port):
    s = socket.socket()
    s.settimeout(2)
    try:
        s.connect(("127.0.0.1", port))
        return True
    except Exception:
        return False
    finally:
        s.close()

if port_open(7687):
    print("Relate ya escucha en el puerto 7687")
else:
    print("arrancando `relate serve` en segundo plano...")
    subprocess.Popen([RELATE_BIN, "serve", "--database", DB, "--bolt", "7687",
                      "--auth", "none"], stdout=LOG, stderr=subprocess.STDOUT,
                     preexec_fn=os.setsid)
    for _ in range(30):
        if port_open(7687):
            break
        time.sleep(2)
    print("Relate escucha en 7687:", port_open(7687))
    if not port_open(7687):
        print("Revisa relate.log")

## Cargar esquema + grafo por Bolt y verificar conteos

Corre `pipeline/load_relate.py` tal cual (driver `neo4j` de Python, Bolt a localhost:7687). Esperado, verificado: **27 nodos, 39 relaciones** (31 semilla + 8 `CRECIO_EN` derivadas). La carga es idempotente: correrla dos veces no duplica nada.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "load_relate.py"], cwd=REPO + "/pipeline")
print("exit:", r.returncode)

## Correr las 12 preguntas en Cypher

Con el CLI, sin pasar por el servidor: `relate -d <store> query` ejecuta cada consulta de `queries.cypher` contra el store. Separa el archivo por consultas (cada bloque `MATCH ... RETURN ...;`) y ejecuta una por una, imprimiendo la respuesta esperada al lado para comparar a ojo.

In [ ]:
import os, re, subprocess

RELATE_BIN = os.path.expanduser("~/.local/bin/relate")
DB = REPO + "/pipeline/data/femsa.rl"
text = open(REPO + "/pipeline/queries.cypher").read()

# Un bloque por pregunta: cada una empieza con su comentario `// Qn.`
# (la Q9 lleva DOS clausulas MATCH en una sola consulta: el regex toma
# del primer MATCH hasta el `;` final, asi que va junta).
blocks = re.split(r"(?=^// Q\d+\.)", text, flags=re.M)
n = 0
for b in blocks:
    m = re.search(r"MATCH.*?;", b, re.S)
    if not m:
        continue
    q = re.search(r"// (Q\d+\..+)", b)
    esp = re.search(r"Esperado: (.+)", b)
    p = subprocess.run([RELATE_BIN, "-d", DB, "query"], input=m.group(0).encode(),
                       capture_output=True)
    rows = [l for l in p.stdout.decode(errors="replace").splitlines() if l.startswith("{")]
    print((q.group(1).strip()[:90] if q else "consulta"), "| esperado:",
          (esp.group(1).strip() if esp else "?"))
    print("   ->", rows[0][:220] if rows else "(sin filas)")
    n += 1
print(f"\n{n} consultas ejecutadas (esperado: 12)")

## Alternativa sin servidor

Si el servidor no levanta, el grafo se puede cargar directamente con el CLI, sin Bolt (mismos archivos, mismo resultado):

```bash
relate -d ./pipeline/data/femsa.rl query < pipeline/schema.cypher
relate -d ./pipeline/data/femsa.rl query < pipeline/load_graph.cypher
```

desde el directorio del repo.

## Guardar en Drive (final)

Todo lo que produjo este notebook vive bajo el repo clonado en Drive (`REPO`), así que sobrevive aunque la sesión de Colab se corte. Esta celda lo verifica y lo lista: si algo falta, lo dice aquí, no después.

El store `femsa.rl` queda en Drive: el notebook 04 (MEMO) no lo necesita, pero cualquier corrida futura del grafo lo reusa sin recargar. Si la sesión se corta, basta re-correr las celdas de arranque del servidor.

In [ ]:
import os, shutil, time


def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.0f} {u}"
        n /= 1024
    return f"{n:.1f} GB"


def _list_dir(path):
    total, nfiles = 0, 0
    for root, _dirs, files in os.walk(path):
        for fn in sorted(files):
            p = os.path.join(root, fn)
            sz = os.path.getsize(p)
            total += sz
            nfiles += 1
            print(f"    {os.path.relpath(p, REPO)} ({_human(sz)})")
    return nfiles, total


# Logs de la sesion (viven en /content, fuera de Drive): copiarlos al repo.
for _log in ["ollama.log", "ollama_error.log", "relate.log"]:
    if os.path.isfile(_log):
        shutil.copy(_log, os.path.join(REPO, _log))
        print(f"log copiado a Drive: {_log}")

ENTRIES = [('pipeline/data/femsa.rl', 'Store Relate (27 nodos, 39 relaciones)', True), ('pipeline/load_graph.cypher', 'Cypher de carga generado (MERGE idempotentes)', True), ('pipeline/schema.cypher', 'Esquema (constraints)', False), ('relate.log', 'Log del servidor Relate', False)]
total_bytes, missing = 0, []
for rel, label, required in ENTRIES:
    p = os.path.join(REPO, rel)
    if os.path.isdir(p):
        nf, tb = _list_dir(p)
        total_bytes += tb
        print(f"{label}: directorio, {nf} archivos, {_human(tb)}")
    elif os.path.isfile(p):
        sz = os.path.getsize(p)
        total_bytes += sz
        mt = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
        print(f"{label}: {rel} ({_human(sz)}, {mt})")
    else:
        status = "FALTA (obligatorio)" if required else "pendiente (opcional)"
        print(f"{label}: {status}: {rel}")
        if required:
            missing.append(rel)
print(f"\nGUARDADO EN DRIVE: {_human(total_bytes)} verificados bajo {REPO}")
if missing:
    print("OJO, faltan obligatorios:", missing)